# Building a Tool-Using AI Agent

## Overview

Unlike the RAG notebook, this notebook allows the LLM to decide **which tool** it needs to answer a user's request.

The agent can:

- Search a PDF
- Get the weather
- Write study notes

Instead of hardcoding the workflow, the model chooses the appropriate tool and uses the returned result to answer the user.

---

## Architecture

```
                User
                  |
                  v
          +------------+
          |   Agent    |
          +-----+------+
                |
      +---------+---------+ 
      v         v         v
 Search PDF  Weather   Write File
      |         |         |
      +---------+---------+
                |
                v
          Final Response
```

In [ ]:
from pathlib import Path
from pypdf import PdfReader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import numpy as np
import requests
import json
import ollama

## Load the PDF

In [ ]:
PDF_PATH = Path("assets/claude_certification_foundation_associate.pdf")

reader = PdfReader(PDF_PATH)

document = ""

for page in reader.pages:
    text = page.extract_text()
    if text:
        document += text + "\n"

In [ ]:
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0

    while start < len(text):
        chunks.append(text[start:start+chunk_size])
        start += chunk_size - overlap

    return chunks

chunks = chunk_text(document)

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english")
vectors = vectorizer.fit_transform(chunks)

# Tool 1

Search the PDF

In [ ]:
def search_pdf(question, top_k=3):
    query = vectorizer.transform([question])

    similarities = cosine_similarity(query, vectors).flatten()

    indices = similarities.argsort()[::-1][:top_k]

    return "\n\n".join(chunks[i] for i in indices)

# Tool 2

Current Weather

In [ ]:
def get_cardiff_weather():
    url = (
        "https://api.open-meteo.com/v1/forecast"
        "?latitude=51.4816"
        "&longitude=-3.1791"
        "&current=temperature_2m,wind_speed_10m"
    )

    data = requests.get(url, timeout=20).json()

    current = data["current"]

    return (
        f"Temperature: {current['temperature_2m']}°C\n"
        f"Wind Speed: {current['wind_speed_10m']} km/h"
    )

# Tool 3

Write Markdown File

In [ ]:
def write_study_note(content):
    path = Path("study_note.md")

    path.write_text(content, encoding="utf-8")

    return f"Saved to {path.resolve()}"

## Tool Schemas

The LLM receives these JSON schemas so it understands what tools are available and how to call them.

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_pdf",
            "description": "Search the certification PDF",
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {
                        "type": "string"
                    }
                },
                "required": ["question"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_cardiff_weather",
            "description": "Get the current weather in Cardiff",
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "write_study_note",
            "description": "Save study notes as a Markdown file",
            "parameters": {
                "type": "object",
                "properties": {
                    "content": {
                        "type": "string"
                    }
                },
                "required": ["content"]
            }
        }
    }
]

## Agent System Prompt

In [ ]:
SYSTEM_PROMPT = """
You are an AI study assistant.

You have access to tools.

Whenever additional information is required, call the appropriate tool.

Never invent information that can be retrieved.

When asked to create notes, use the write_study_note tool.
"""

In [ ]:
FUNCTIONS = {
    "search_pdf": search_pdf,
    "get_cardiff_weather": get_cardiff_weather,
    "write_study_note": write_study_note,
}

In [ ]:
messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    }
]

In [ ]:
user_question = (
    "Summarize the certification and "
    "include today's weather in Cardiff."
)

messages.append({
    "role": "user",
    "content": user_question
})

In [ ]:
response = ollama.chat(
    model="qwen3",
    messages=messages,
    tools=TOOLS
)

## Execute Tool Calls

In [ ]:
tool_calls = response["message"].get("tool_calls", [])

for tool_call in tool_calls:
    name = tool_call["function"]["name"]
    args = tool_call["function"].get("arguments", {})

    result = FUNCTIONS[name](**args)

    messages.append({
        "role": "tool",
        "name": name,
        "content": str(result)
    })

In [ ]:
final = ollama.chat(
    model="qwen3",
    messages=messages
)

print(final["message"]["content"])

## Example Prompts

Try:

- Summarize Chapter 2.
- What topics are covered in the exam?
- What is today's weather in Cardiff?
- Create study notes for the PDF.
- Save the study guide as Markdown.